# Train inverse solvers
In this notebook, we train models to reproduce the initial condition from the final state individually in the case of each vortex pattern (same sign pair, opposite sign pair, same sign triple, mixed sign triple).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import itertools
from torch.utils.data import TensorDataset, DataLoader, random_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load data
Four separate inverse models will be trained, one for each vortex family. Each model takes the evolved vorticity field $\omega(T)$ (or `omegaT`) as input and attempts to recover the parameters of the corresponding initial vortex configuration.

For each family, the dataset therefore provides three pieces of information used during training. They are the evolved field $\omega(T)$, the true initial field $\omega(0)$ (or `omega0`), and the parameters which describe the initial vortices.

In [2]:
same_pair_data = np.load("vortex_same_sign_pair_50000_aug.npz")
opposite_pair_data = np.load("vortex_opposite_sign_pair_50000_aug.npz")
same_triple_data = np.load("vortex_same_sign_triple_50000_aug.npz")
mixed_triple_data = np.load("vortex_mixed_sign_triple_50000_aug.npz")

omega0_sp = same_pair_data["omega0"]
omegaT_sp = same_pair_data["omegaT"]
params_sp = same_pair_data["parameters"][:, :2, :] # eliminate unnecessary third row from parameter data since we only have two vortices here

omega0_op = opposite_pair_data["omega0"]
omegaT_op = opposite_pair_data["omegaT"]
params_op = opposite_pair_data["parameters"][:, :2, :] # again, eliminate the artifical third parameter row for vortex pair data

omega0_st = same_triple_data["omega0"]
omegaT_st = same_triple_data["omegaT"]
params_st = same_triple_data["parameters"]

omega0_mt = mixed_triple_data["omega0"]
omegaT_mt = mixed_triple_data["omegaT"]
params_mt = mixed_triple_data["parameters"]

# Rendering vortices from parameters
Rather than directly predicting all $128\times 128$ values of the initial vorticity field, the inverse models predict the low-dimensional parameters of the initial vortex configuration. Each vortex is described by its center $(x_0,y_0)$, amplitude, and width $\sigma$.

The function below maps a predicted collection of vortex parameters to an actual vortex field. Because this rendering is implemented with PyTorch operations, it remains differentiable and can therefore be used directly as a part of the training loss.

In [3]:
N = 128
L = np.pi

x_grid = torch.linspace(-L/2, L/2, N+1)[:-1]
y_grid = torch.linspace(-L/2, L/2, N+1)[:-1]

Y, X = torch.meshgrid(y_grid, x_grid, indexing="ij")

X = X.to(device)
Y = Y.to(device)

def periodic_diff(x1, x0, L):
    return (x1 - x0 + L/2) % L - L/2

def render_gaussian_vortices(params, X, Y, L):
    x0 = params[:, :, 0]
    y0 = params[:, :, 1]
    amp = params[:, :, 2]
    sigma = params[:, :, 3]

    theta_x = 2 * torch.pi * (X[None, None, :, :] - x0[:, :, None, None]) / L
    theta_y = 2 * torch.pi * (Y[None, None, :, :] - y0[:, :, None, None]) / L

    sigma_theta = (2 * torch.pi / L) * sigma[:, :, None, None]

    exponent = (-2.0 + torch.cos(theta_x) + torch.cos(theta_y)) / (sigma_theta**2)

    vortices = amp[:, :, None, None] * torch.exp(exponent)

    omega = vortices.sum(dim=1)

    return omega

# Batch loaders
Each dataset is normalized by the maximum absolute value of its evolved fields, split into training and test subsets, and loaded in batches of 64.

In [4]:
def make_batch_loaders(omegaT, omega0, params, batch_size=64, train_frac=0.8):
    # returns train loader and test loader

    X = torch.as_tensor(omegaT, dtype=torch.float32)
    y_params = torch.as_tensor(params, dtype=torch.float32)
    y_omega0 = torch.as_tensor(omega0, dtype=torch.float32)

    # normalize input image
    input_scale = X.abs().max()
    X = X / input_scale

    # add a dimension for Conv2d
    X = X[:,None,:,:]

    dataset = TensorDataset(X, y_params, y_omega0)
    n_total = len(dataset)
    n_train = int(train_frac * n_total)
    n_test = n_total - n_train

    train_dataset, test_dataset = random_split(
        dataset,
        [n_train, n_test],
        generator=torch.Generator().manual_seed(0)
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size = batch_size,
        shuffle = True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size = batch_size,
        shuffle = False
    )

    return train_loader, test_loader, input_scale

In [5]:
# Make loaders for same sign pairs
train_loader_sp, test_loader_sp, input_scale_sp = make_batch_loaders(
    omegaT_sp,
    omega0_sp,
    params_sp,
    batch_size=64
)

# Make loaders for opposite sign pairs
train_loader_op, test_loader_op, input_scale_op = make_batch_loaders(
    omegaT_op,
    omega0_op,
    params_op,
   batch_size=64
)

# Make loaders for same sign triples
train_loader_st, test_loader_st, input_scale_st = make_batch_loaders(
    omegaT_st,
    omega0_st,
    params_st,
    batch_size=64
)

# Make loaders for mixed sign triples
train_loader_mt, test_loader_mt, input_scale_mt = make_batch_loaders(
    omegaT_mt,
    omega0_mt,
    params_mt,
    batch_size=64
)

# Inverse model setup
The inverse networks use convolutional layers to determine spatial features from the evolved vorticity field, using circular padding to account for our periodic domain. Since vortex pairs and vortex triples require different numbers of output parameters, we use separate architectures for pairs and triples.

The pair model outputs two sets of four vortex parameters, while the triple model outputs three sets. The raw network outputs are transformed so that predicted vortex centers, amplitudes, and widths remain essentially within the parameter ranges used to generate the data.

In [24]:
# First we define a classes to predict parameters for pairs of vortices and for triples of vortices

class PairInverseNet(nn.Module):
    def __init__(self, L=np.pi, amp_max=15.0, sigma_min=0.10, sigma_max=0.16):
        super().__init__()

        self.L = L
        self.amp_max = amp_max
        self.sigma_min = sigma_min
        self.sigma_max = sigma_max

        self.model = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=5, padding=2, padding_mode="circular"),
            nn.ReLU(),
            nn.AvgPool2d(2),

            nn.Conv2d(16, 32, kernel_size=5, padding=2, padding_mode="circular"),
            nn.ReLU(),
            nn.AvgPool2d(2),

            nn.Conv2d(32, 64, kernel_size=5, padding=2, padding_mode="circular"),
            nn.ReLU(),
            nn.AvgPool2d(2),

            nn.Conv2d(64, 64, kernel_size=5, padding=2, padding_mode="circular"),
            nn.ReLU(),
            nn.AvgPool2d(2),
        )

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * (N // 16)**2, 128),
            nn.Tanh(),
            nn.Linear(128, 8) # Returns 4 * 2 = 8 parameter slots: 4 parameters ((x0, y0), amplitude, sigma), for each of 2 vortices
        )

    def forward(self, x):
        z = self.model(x)
        raw = self.head(z)
        
        raw = raw.view(-1, 2, 4)

        raw_xy = raw[:, :, 0:2]
        raw_amp = raw[:, :, 2]
        raw_sigma = raw[:, :, 3]

        xy = (self.L / 2) * torch.tanh(raw_xy)

        amp = self.amp_max * torch.tanh(raw_amp)

        sigma = self.sigma_min + (self.sigma_max - self.sigma_min) * torch.sigmoid(raw_sigma)

        params = torch.zeros_like(raw)
        params[:, :, 0:2] = xy
        params[:, :, 2] = amp
        params[:, :, 3] = sigma

        return params

class TripleInverseNet(nn.Module):
    def __init__(self, L=np.pi, amp_max=15.0, sigma_min=0.10, sigma_max=0.16):
        super().__init__()

        self.L = L
        self.amp_max = amp_max
        self.sigma_min = sigma_min
        self.sigma_max = sigma_max

        self.model = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=5, padding=2, padding_mode="circular"),
            nn.ReLU(),
            nn.AvgPool2d(2),

            nn.Conv2d(16, 32, kernel_size=5, padding=2, padding_mode="circular"),
            nn.ReLU(),
            nn.AvgPool2d(2),

            nn.Conv2d(32, 64, kernel_size=5, padding=2, padding_mode="circular"),
            nn.ReLU(),
            nn.AvgPool2d(2),

            nn.Conv2d(64, 64, kernel_size=5, padding=2, padding_mode="circular"),
            nn.ReLU(),
            nn.AvgPool2d(2),
            
            nn.Flatten(),
            nn.Linear(64 * (N // 16)**2, 128),
            nn.Tanh(),
            nn.Linear(128, 12) # Returns 4 * 3 = 12 parameter slots: 4 parameters ((x0, y0), amplitude, sigma), for each of 3 vortices
        )

    def forward(self, x):
        raw = self.model(x)
        raw = raw.view(-1, 3, 4)

        raw_xy = raw[:, :, 0:2]
        raw_amp = raw[:, :, 2]
        raw_sigma = raw[:, :, 3]

        xy = (self.L / 2) * torch.tanh(raw_xy)

        amp = self.amp_max * torch.tanh(raw_amp)

        sigma = self.sigma_min + (self.sigma_max - self.sigma_min) * torch.sigmoid(raw_sigma)

        params = torch.zeros_like(raw)
        params[:, :, 0:2] = xy
        params[:, :, 2] = amp
        params[:, :, 3] = sigma

        return params

The inverse models are trained using a loss that measures two related but separate things: parameter accuracy and reconstruction accuracy. The parameter component compares the predicted vortex centers, amplitudes, and widths with their true values, while the field component renders the predicted parameters into an initial vorticity field and compares it directly with the true field $\omega(0)$.

Because the vortices have no intrinsic ordering, the parameter loss must also be permutation invariant. For a pair, the two possible assignments between predicted and true vortices are compared. For a triple, all six possible assignments are considered. For each sample, the assignment with the smallest error for the *center parameters* is selected, and the amplitude and width errors are evaluated using that same assignment.

Center errors are computed using periodic coordinate differences, so that two vortex centers lying near opposite edges of the computational domain can still be recognized as spatially close on the torus. Amplitude and width errors are normalized by 15 and 0.06, respectively.

We then define the final training loss to be a weighted combination of the four losses:

$\mathcal{L}_{\mathrm{train}}=5\mathcal{L}_{\mathrm{field}}+\mathcal{L}_{\mathrm{center}}+\mathcal{L}_{\mathrm{amp}}+\mathcal{L}_{\sigma},$

where $\mathcal{L}_{\mathrm{field}}$ is the mean squared error between the rendered prediction of the initial vorticity field and the true initial vorticity field.

In [7]:
def matched_center_amp_sigma_loss_2(params_pred, params_true):
    """
    Permutation-invariant center + amplitude + sigma loss for vortex pairs.

    params_pred and params_true have shape (B, 2, 4).
    """

    pred_xy = params_pred[:, :, 0:2]
    true_xy = params_true[:, :, 0:2]

    pred_amp = params_pred[:, :, 2]
    true_amp = params_true[:, :, 2]

    pred_sigma = params_pred[:, :, 3]
    true_sigma = params_true[:, :, 3]

    # matching 1: pred 0 <-> true 0, pred 1 <-> true 1

    dxy_id = periodic_diff(pred_xy, true_xy, L)
    center_loss_id = torch.mean(torch.sum(dxy_id**2, dim=2), dim=1)

    amp_loss_id = torch.mean(((pred_amp - true_amp) / 15.0)**2, dim=1)
    sigma_loss_id = torch.mean(((pred_sigma - true_sigma) / 0.06)**2, dim=1)

    # matching 2: pred 0 <-> true 1, pred 1 <-> true 0

    true_xy_swap = true_xy[:, [1, 0], :]
    true_amp_swap = true_amp[:, [1, 0]]
    true_sigma_swap = true_sigma[:, [1, 0]]

    dxy_swap = periodic_diff(pred_xy, true_xy_swap, L)
    center_loss_swap = torch.mean(torch.sum(dxy_swap**2, dim=2), dim=1)

    amp_loss_swap = torch.mean(((pred_amp - true_amp_swap) / 15.0)**2, dim=1)
    sigma_loss_swap = torch.mean(((pred_sigma - true_sigma_swap) / 0.06)**2, dim=1)

    # choose better matching per sample, based on center error

    use_swap = center_loss_swap < center_loss_id

    center_loss = torch.where(use_swap, center_loss_swap, center_loss_id).mean()
    amp_loss = torch.where(use_swap, amp_loss_swap, amp_loss_id).mean()
    sigma_loss = torch.where(use_swap, sigma_loss_swap, sigma_loss_id).mean()

    return center_loss, amp_loss, sigma_loss


def combined_loss_2(
    params_pred,
    params_true,
    omega0_true,
    field_weight=5.0,
    center_weight=1.0,
    amp_weight=1.0,
    sigma_weight=1.0
):
    omega0_pred = render_gaussian_vortices(params_pred, X, Y, L)

    field_loss = torch.mean((omega0_pred - omega0_true)**2)

    center_loss, amp_loss, sigma_loss = matched_center_amp_sigma_loss_2(params_pred, params_true)

    loss = (
        field_weight * field_loss
        + center_weight * center_loss
        + amp_weight * amp_loss
        + sigma_weight * sigma_loss
    )

    return loss, field_loss, center_loss, amp_loss, sigma_loss

def matched_center_amp_sigma_loss_3(params_pred, params_true):
    """
    Permutation-invariant center + amplitude + sigma loss for vortex triples.

    params_pred and params_true have shape (B, 3, 4).

    Each vortex has:
        params[:, :, 0] = x center
        params[:, :, 1] = y center
        params[:, :, 2] = amplitude
        params[:, :, 3] = sigma
    """

    pred_xy = params_pred[:, :, 0:2]
    true_xy = params_true[:, :, 0:2]

    pred_amp = params_pred[:, :, 2]
    true_amp = params_true[:, :, 2]

    pred_sigma = params_pred[:, :, 3]
    true_sigma = params_true[:, :, 3]

    center_losses = []
    amp_losses = []
    sigma_losses = []

    # There are 6 possible matchings for 3 vortices.
    perms = list(itertools.permutations([0, 1, 2]))

    for perm in perms:
        # Reorder the true vortices according to this proposed matching.
        true_xy_perm = true_xy[:, list(perm), :]
        true_amp_perm = true_amp[:, list(perm)]
        true_sigma_perm = true_sigma[:, list(perm)]

        # Center loss for each sample in the batch.
        dxy = periodic_diff(pred_xy, true_xy_perm, L)
        center_loss_perm = torch.mean(torch.sum(dxy**2, dim=2), dim=1)

        # Amplitude and sigma losses for each sample in the batch.
        amp_loss_perm = torch.mean(((pred_amp - true_amp_perm) / 15.0)**2, dim=1)
        sigma_loss_perm = torch.mean(((pred_sigma - true_sigma_perm) / 0.06)**2, dim=1)

        center_losses.append(center_loss_perm)
        amp_losses.append(amp_loss_perm)
        sigma_losses.append(sigma_loss_perm)

    # Shape: (6, B)
    center_losses = torch.stack(center_losses, dim=0)
    amp_losses = torch.stack(amp_losses, dim=0)
    sigma_losses = torch.stack(sigma_losses, dim=0)

    # For each sample, choose the permutation with the smallest center error.
    # Shape: (B,)
    best_perm = torch.argmin(center_losses, dim=0)

    # Batch indices: 0, 1, 2, ..., B-1
    B = params_pred.shape[0]
    batch_indices = torch.arange(B, device=params_pred.device)

    # Pick the center/amplitude/sigma losses using the best permutation per sample.
    center_loss = center_losses[best_perm, batch_indices].mean()
    amp_loss = amp_losses[best_perm, batch_indices].mean()
    sigma_loss = sigma_losses[best_perm, batch_indices].mean()

    return center_loss, amp_loss, sigma_loss

def combined_loss_3(
    params_pred,
    params_true,
    omega0_true,
    field_weight=5.0,
    center_weight=1.0,
    amp_weight=1.0,
    sigma_weight=1.0
):
    omega0_pred = render_gaussian_vortices(params_pred, X, Y, L)

    field_loss = torch.mean((omega0_pred - omega0_true)**2)

    center_loss, amp_loss, sigma_loss = matched_center_amp_sigma_loss_3(params_pred, params_true)

    loss = (
        field_weight * field_loss
        + center_weight * center_loss
        + amp_weight * amp_loss
        + sigma_weight * sigma_loss
    )

    return loss, field_loss, center_loss, amp_loss, sigma_loss

# Reconstruction error
In addition to the individual components of the training loss, model performance on the held out test data is evaluated directly in terms of the reconstructed initial vorticity field. The function below reports both the mean squared field error and a global relative $L^2$ reconstruction error, computed by pooling the squared error and true field norms over the full test set before taking their ratio.

In [8]:
# Now, let us define a function to evaluate error for a given model
def evaluate_field_error(model, loader):
    model.eval()

    error_L2_norm_sq = 0.0
    true_L2_norm_sq = 0.0

    total_num_images = len(loader.dataset)

    with torch.no_grad():
        for xb, _, omega0_b in loader:
            xb = xb.to(device)
            omega0_b = omega0_b.to(device)

            params_pred = model(xb)

            omega0_pred = render_gaussian_vortices(params_pred, X, Y, L)

            error = omega0_pred - omega0_b

            error_L2_norm_sq += torch.sum(error**2).item()
            true_L2_norm_sq += torch.sum(omega0_b**2).item()

    mse = error_L2_norm_sq / (total_num_images * N**2)
    rel_L2_error = (error_L2_norm_sq / true_L2_norm_sq)**0.5

    return mse, rel_L2_error

# Training pair inverse solvers
The same training procedure is used for both pair classes. At each epoch, the network predicts vortex parameters from the evolved fields, and the combined loss is used to update the model parameters.

The reported training quantities separate the total loss into its field, center, amplitude, and width components. Performance on the test set is evaluated using the reconstructed initial field, with both mean squared error and relative $L^2$ error reported after each epoch.

In [9]:
# Here we define a training function for models for pairs of vortices
def train_model_2(
    model,
    train_loader,
    test_loader,
    optimizer,
    learning_rate,
    start_epoch,
    additional_epochs,
):
    # update optimizer learning rate
    for param_group in optimizer.param_groups:
        param_group["lr"] = learning_rate

    for epoch in range(start_epoch, start_epoch + additional_epochs):
        model.train()

        total_train_loss = 0.0
        total_field_loss = 0.0
        total_center_loss = 0.0
        total_amp_loss = 0.0
        total_sigma_loss = 0.0
        total_num = 0

        for xb, params_b, omega0_b in train_loader:
            xb = xb.to(device)
            params_b = params_b.to(device)
            omega0_b = omega0_b.to(device)

            params_pred = model(xb)

            loss, field_loss, center_loss, amp_loss, sigma_loss = combined_loss_2(
                params_pred,
                params_b,
                omega0_b,
                field_weight=5.0,
                center_weight=1.0,
                amp_weight=1.0,
                sigma_weight=1.0,
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            batch_size = xb.shape[0]

            total_train_loss += loss.item() * batch_size
            total_field_loss += field_loss.item() * batch_size
            total_center_loss += center_loss.item() * batch_size
            total_amp_loss += amp_loss.item() * batch_size
            total_sigma_loss += sigma_loss.item() * batch_size
            total_num += batch_size

        avg_train_loss = total_train_loss / total_num
        avg_field_loss = total_field_loss / total_num
        avg_center_loss = total_center_loss / total_num
        avg_amp_loss = total_amp_loss / total_num
        avg_sigma_loss = total_sigma_loss / total_num

        test_mse, test_rel_l2 = evaluate_field_error(model, test_loader)

        print(
            f"epoch {epoch:3d} | "
            f"train {avg_train_loss:.5f} | "
            f"field {avg_field_loss:.5f} | "
            f"center {avg_center_loss:.5f} | "
            f"amp {avg_amp_loss:.5f} | "
            f"sigma {avg_sigma_loss:.5f} | "
            f"testMSE {test_mse:.5f} | "
            f"testRelL2 {test_rel_l2:.4f}"
        )

# Same sign pair training

In [10]:
model_sp = PairInverseNet(L=L).to(device)
optimizer_sp = torch.optim.Adam(model_sp.parameters(), lr=1e-4)

In [11]:
train_model_2(
    model_sp,
    train_loader_sp,
    test_loader_sp,
    optimizer_sp,
    1e-4,
    0,
    150)

epoch   0 | train 3.79206 | field 0.68946 | center 0.06306 | amp 0.14084 | sigma 0.14088 | testMSE 0.42208 | testRelL2 0.4522
epoch   1 | train 2.24589 | field 0.38979 | center 0.01994 | amp 0.08118 | sigma 0.19581 | testMSE 0.37212 | testRelL2 0.4246
epoch   2 | train 1.88071 | field 0.32688 | center 0.01483 | amp 0.06319 | sigma 0.16828 | testMSE 0.26349 | testRelL2 0.3573
epoch   3 | train 1.40326 | field 0.24336 | center 0.00917 | amp 0.04473 | sigma 0.13256 | testMSE 0.23137 | testRelL2 0.3348
epoch   4 | train 1.25222 | field 0.21749 | center 0.00796 | amp 0.03908 | sigma 0.11773 | testMSE 0.21031 | testRelL2 0.3192
epoch   5 | train 1.15944 | field 0.20110 | center 0.00729 | amp 0.03676 | sigma 0.10990 | testMSE 0.19372 | testRelL2 0.3063
epoch   6 | train 1.07163 | field 0.18565 | center 0.00668 | amp 0.03443 | sigma 0.10226 | testMSE 0.17517 | testRelL2 0.2913
epoch   7 | train 0.99184 | field 0.17199 | center 0.00613 | amp 0.03204 | sigma 0.09372 | testMSE 0.16465 | testRelL2

The same sign pair model improves steadily over 150 epochs, with the global test relative $L^2$ reconstruction error falling from about 45% initially to roughly 7–8% near the end of training. The test error fluctuates somewhat from epoch to epoch even as the training loss continues to decrease.

# Opposite sign pair training

In [12]:
model_op = PairInverseNet(L=L).to(device)
optimizer_op = torch.optim.Adam(model_op.parameters(), lr=1e-4)

In [13]:
train_model_2(
    model_op,
    train_loader_op,
    test_loader_op,
    optimizer_op,
    1e-4,
    0,
    150)

epoch   0 | train 4.30161 | field 0.77059 | center 0.07558 | amp 0.28667 | sigma 0.08640 | testMSE 0.35899 | testRelL2 0.4999
epoch   1 | train 1.32081 | field 0.23999 | center 0.00695 | amp 0.04698 | sigma 0.06692 | testMSE 0.17974 | testRelL2 0.3537
epoch   2 | train 0.78370 | field 0.13991 | center 0.00348 | amp 0.02573 | sigma 0.05492 | testMSE 0.11075 | testRelL2 0.2777
epoch   3 | train 0.56013 | field 0.09856 | center 0.00229 | amp 0.01556 | sigma 0.04951 | testMSE 0.08714 | testRelL2 0.2463
epoch   4 | train 0.43951 | field 0.07638 | center 0.00175 | amp 0.01053 | sigma 0.04532 | testMSE 0.07421 | testRelL2 0.2273
epoch   5 | train 0.39634 | field 0.06874 | center 0.00154 | amp 0.00960 | sigma 0.04149 | testMSE 0.06480 | testRelL2 0.2124
epoch   6 | train 0.36161 | field 0.06273 | center 0.00138 | amp 0.00914 | sigma 0.03742 | testMSE 0.06231 | testRelL2 0.2083
epoch   7 | train 0.32350 | field 0.05604 | center 0.00122 | amp 0.00882 | sigma 0.03326 | testMSE 0.05580 | testRelL2

The opposite sign pair inverse problem is learned more accurately. After 150 epochs, the global test relative $L^2$ error has fallen to approximately 3%, substantially below that of the same-sign pair model.

# Training triple inverse solvers
The triple models are trained with the same overall procedure as the pair models, but predict three vortices rather than two.

As before, training progress is monitored through each of the individual loss components. The main measure of performance is the accuracy of the reconstructed initial vorticity field.

In [14]:
# Here we define a function for training inverse solvers for vortex triples
def train_model_3(
    model,
    train_loader,
    test_loader,
    optimizer,
    learning_rate,
    start_epoch,
    additional_epochs,
):
    # update optimizer learning rate
    for param_group in optimizer.param_groups:
        param_group["lr"] = learning_rate

    for epoch in range(start_epoch, start_epoch + additional_epochs):
        model.train()

        total_train_loss = 0.0
        total_field_loss = 0.0
        total_center_loss = 0.0
        total_amp_loss = 0.0
        total_sigma_loss = 0.0
        total_num = 0

        for xb, params_b, omega0_b in train_loader:
            xb = xb.to(device)
            params_b = params_b.to(device)
            omega0_b = omega0_b.to(device)

            params_pred = model(xb)

            loss, field_loss, center_loss, amp_loss, sigma_loss = combined_loss_3(
                params_pred,
                params_b,
                omega0_b,
                field_weight=5.0,
                center_weight=1.0,
                amp_weight=1.0,
                sigma_weight=1.0,
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            batch_size = xb.shape[0]

            total_train_loss += loss.item() * batch_size
            total_field_loss += field_loss.item() * batch_size
            total_center_loss += center_loss.item() * batch_size
            total_amp_loss += amp_loss.item() * batch_size
            total_sigma_loss += sigma_loss.item() * batch_size
            total_num += batch_size

        avg_train_loss = total_train_loss / total_num
        avg_field_loss = total_field_loss / total_num
        avg_center_loss = total_center_loss / total_num
        avg_amp_loss = total_amp_loss / total_num
        avg_sigma_loss = total_sigma_loss / total_num

        test_mse, test_rel_l2 = evaluate_field_error(model, test_loader)

        print(
            f"epoch {epoch:3d} | "
            f"train {avg_train_loss:.5f} | "
            f"field {avg_field_loss:.5f} | "
            f"center {avg_center_loss:.5f} | "
            f"amp {avg_amp_loss:.5f} | "
            f"sigma {avg_sigma_loss:.5f} | "
            f"testMSE {test_mse:.5f} | "
            f"testRelL2 {test_rel_l2:.4f}"
        )

# Same sign triple training

In [15]:
model_st = TripleInverseNet(L=L).to(device)
optimizer_st = torch.optim.Adam(model_st.parameters(), lr=1e-4)

In [16]:
train_model_3(
    model_st,
    train_loader_st,
    test_loader_st,
    optimizer_st,
    1e-4,
    0,
    150)

epoch   0 | train 3.96272 | field 0.74054 | center 0.04528 | amp 0.09438 | sigma 0.12039 | testMSE 0.33994 | testRelL2 0.3079
epoch   1 | train 1.84303 | field 0.32350 | center 0.01496 | amp 0.04707 | sigma 0.16348 | testMSE 0.31397 | testRelL2 0.2959
epoch   2 | train 1.73463 | field 0.30207 | center 0.01458 | amp 0.04704 | sigma 0.16266 | testMSE 0.29786 | testRelL2 0.2882
epoch   3 | train 1.69717 | field 0.29483 | center 0.01446 | amp 0.04736 | sigma 0.16122 | testMSE 0.29212 | testRelL2 0.2854
epoch   4 | train 1.68442 | field 0.29278 | center 0.01449 | amp 0.04738 | sigma 0.15867 | testMSE 0.28831 | testRelL2 0.2835
epoch   5 | train 1.66850 | field 0.29013 | center 0.01442 | amp 0.04712 | sigma 0.15633 | testMSE 0.28692 | testRelL2 0.2828
epoch   6 | train 1.65437 | field 0.28792 | center 0.01433 | amp 0.04664 | sigma 0.15380 | testMSE 0.28698 | testRelL2 0.2829
epoch   7 | train 1.65156 | field 0.28821 | center 0.01426 | amp 0.04606 | sigma 0.15021 | testMSE 0.29274 | testRelL2

The same sign triple reconstruction problem is more difficult than either pair problem. Over 150 epochs, the global test relative $L^2$ error decreases to approximately 13%. The steady improvement throughout training indicates that the model continues to extract useful information about the three-vortex initial configuration, although reconstruction remains less accurate than for the pair models.

# Mixed sign triple training

In [17]:
model_mt = TripleInverseNet(L=L).to(device)
optimizer_mt = torch.optim.Adam(model_mt.parameters(), lr=1e-4)

In [18]:
train_model_3(
    model_mt,
    train_loader_mt,
    test_loader_mt,
    optimizer_mt,
    1e-4,
    0,
    150)

epoch   0 | train 8.12796 | field 1.47308 | center 0.11341 | amp 0.55184 | sigma 0.09731 | testMSE 1.06403 | testRelL2 0.6790
epoch   1 | train 4.96346 | field 0.89764 | center 0.03376 | amp 0.31885 | sigma 0.12265 | testMSE 0.78887 | testRelL2 0.5847
epoch   2 | train 3.67626 | field 0.66506 | center 0.02063 | amp 0.20756 | sigma 0.12275 | testMSE 0.57817 | testRelL2 0.5005
epoch   3 | train 3.00783 | field 0.54251 | center 0.01580 | amp 0.15117 | sigma 0.12834 | testMSE 0.49716 | testRelL2 0.4641
epoch   4 | train 2.62760 | field 0.47126 | center 0.01361 | amp 0.12496 | sigma 0.13273 | testMSE 0.44579 | testRelL2 0.4395
epoch   5 | train 2.41717 | field 0.43174 | center 0.01255 | amp 0.11098 | sigma 0.13495 | testMSE 0.41171 | testRelL2 0.4224
epoch   6 | train 2.26609 | field 0.40314 | center 0.01183 | amp 0.10346 | sigma 0.13511 | testMSE 0.39478 | testRelL2 0.4136
epoch   7 | train 2.15157 | field 0.38163 | center 0.01133 | amp 0.09792 | sigma 0.13419 | testMSE 0.37157 | testRelL2

In [20]:
train_model_3(
    model_mt,
    train_loader_mt,
    test_loader_mt,
    optimizer_mt,
    1e-4,
    150,
    100)

epoch 150 | train 0.30406 | field 0.05370 | center 0.00102 | amp 0.01010 | sigma 0.02443 | testMSE 0.08165 | testRelL2 0.1881
epoch 151 | train 0.30126 | field 0.05319 | center 0.00101 | amp 0.01004 | sigma 0.02427 | testMSE 0.08254 | testRelL2 0.1891
epoch 152 | train 0.29633 | field 0.05232 | center 0.00099 | amp 0.00981 | sigma 0.02394 | testMSE 0.08510 | testRelL2 0.1920
epoch 153 | train 0.29443 | field 0.05197 | center 0.00099 | amp 0.00981 | sigma 0.02377 | testMSE 0.08063 | testRelL2 0.1869
epoch 154 | train 0.29162 | field 0.05148 | center 0.00097 | amp 0.00968 | sigma 0.02354 | testMSE 0.08004 | testRelL2 0.1862
epoch 155 | train 0.28840 | field 0.05092 | center 0.00096 | amp 0.00963 | sigma 0.02323 | testMSE 0.08361 | testRelL2 0.1903
epoch 156 | train 0.28662 | field 0.05062 | center 0.00095 | amp 0.00954 | sigma 0.02306 | testMSE 0.08366 | testRelL2 0.1904
epoch 157 | train 0.28604 | field 0.05060 | center 0.00094 | amp 0.00929 | sigma 0.02279 | testMSE 0.07929 | testRelL2

In [21]:
train_model_3(
    model_mt,
    train_loader_mt,
    test_loader_mt,
    optimizer_mt,
    5e-5,
    250,
    100)

epoch 250 | train 0.12138 | field 0.02091 | center 0.00035 | amp 0.00451 | sigma 0.01197 | testMSE 0.05905 | testRelL2 0.1600
epoch 251 | train 0.12139 | field 0.02093 | center 0.00035 | amp 0.00449 | sigma 0.01188 | testMSE 0.05982 | testRelL2 0.1610
epoch 252 | train 0.12296 | field 0.02126 | center 0.00035 | amp 0.00448 | sigma 0.01185 | testMSE 0.05942 | testRelL2 0.1605
epoch 253 | train 0.12341 | field 0.02137 | center 0.00035 | amp 0.00445 | sigma 0.01178 | testMSE 0.05996 | testRelL2 0.1612
epoch 254 | train 0.12315 | field 0.02132 | center 0.00035 | amp 0.00445 | sigma 0.01175 | testMSE 0.06005 | testRelL2 0.1613
epoch 255 | train 0.12261 | field 0.02123 | center 0.00034 | amp 0.00442 | sigma 0.01168 | testMSE 0.06144 | testRelL2 0.1632
epoch 256 | train 0.12234 | field 0.02119 | center 0.00034 | amp 0.00438 | sigma 0.01165 | testMSE 0.06048 | testRelL2 0.1619
epoch 257 | train 0.12124 | field 0.02098 | center 0.00034 | amp 0.00438 | sigma 0.01160 | testMSE 0.05900 | testRelL2

The mixed sign triple problem proves to be the most difficult of the four inverse problems and is therefore trained for longer. After the initial 150 epochs at a learning rate of $10^{-4}$, the global test relative $L^2$ error remains around 19%. Training is continued for another 100 epochs at the same learning rate, reducing the error to roughly 16.6%. A final 100 epochs are then performed with the learning rate reduced to $5\times 10^{-5}$.

By the end of training, the global test relative $L^2$ error is approximately 15.7%. The slower convergence and larger remaining error indicate that recovering mixed sign triple configurations from their evolved fields is substantially more difficult than the other vortex families considered here.

# Save inverse models
Each trained inverse model is saved together with the input normalization scale used for its corresponding vortex family. Saving this scale ensures that evolved fields can be normalized in the same way when the models are later loaded for inference.

In [22]:
torch.save(
    {
        "model_state_dict": model_sp.state_dict(),
        "input_scale": input_scale_sp.item(),
    },
    "inverse_same_pair.pt",
)

torch.save(
    {
        "model_state_dict": model_op.state_dict(),
        "input_scale": input_scale_op.item(),
    },
    "inverse_opposite_pair.pt",
)

torch.save(
    {
        "model_state_dict": model_st.state_dict(),
        "input_scale": input_scale_st.item(),
    },
    "inverse_same_triple.pt",
)

torch.save(
    {
        "model_state_dict": model_mt.state_dict(),
        "input_scale": input_scale_mt.item(),
    },
    "inverse_mixed_triple.pt",
)

# Summary
Separate inverse models have now been trained for all four vortex families. On their test sets, the final global relative $L^2$ reconstruction errors are approximately 8% for same sign pairs, 3% for opposite sign pairs, 13% for same sign triples, and 16% for mixed sign triples.

The results show a clear increase in difficulty as the vortex configurations become more complex, with mixed-sign triples posing the most challenging inverse problem. The next stage combines these models with the pattern classifier to form the full inverse pipeline and evaluates its performance across all four vortex families.